<a href="https://colab.research.google.com/github/Anshu-kumar-singh/Titanic-Survival-Predication-Basic-Project/blob/main/titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# building connection so we can get data from kagle

In [6]:
pip install kaggle

In [7]:
import os

# Create folder
os.makedirs('/root/.kaggle', exist_ok=True)

# Move kaggle.json
!cp kaggle.json /root/.kaggle/

# Set permission
!chmod 600 /root/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [8]:
!kaggle competitions download -c titanic

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


In [9]:
!unzip titanic.zip

unzip:  cannot find or open titanic.zip, titanic.zip.zip or titanic.zip.ZIP.


# seeing how the data look

In [10]:
import pandas as pd

df = pd.read_csv("train.csv")

df.head() # to read the starting 5 rows

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
print(df['Cabin'].to_string()) # we got to know max value are null and they are random number so they are not usefull
                               # they might help in location tracking bu tfor this situation not needed

# droping useless data


In [ ]:
df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)
# we have droped this column
# PassengerId = this were normal number the where not helping in any way
# Name = nothing to do with survival
# ticket = not getting any pattern in this situation [may be helpfull to get the location but not this time ]
# cabin = upper hai

In [ ]:
df.head() # to read the starting 5 rows

🎯 #1. Survived (Target)

0 = No, 1 = Yes

👉 This is what your model is trying to predict.

🟦 #2. Pclass (Passenger Class)

1 = Rich, 3 = Poor

👉 Strong signal:

1st class → higher survival
3rd class → lower survival

💡 Reason: better cabins, priority rescue

🟨 #3. Sex

male / female

👉 MOST IMPORTANT FEATURE

Females → very high survival
Males → lower survival

💡 “Women and children first” rule

🟩 #4. Age

👉 Helps identify:

Children → higher survival
Elderly → lower survival

⚠️ Has missing values → we’ll fix this next

🟧 #5. SibSp (Siblings/Spouse)

👉 Shows family presence

Small family → better survival
Too many → harder to escape

🟥 #6. Parch (Parents/Children)

👉 Similar to SibSp

💡 Combined insight:

Traveling alone vs with family matters

🟪 #7. Fare

👉 Ticket price → wealth indicator

Higher fare → higher class → better survival
🟫 #8. Embarked
C, Q, S

👉 Location where passenger boarded

Some ports had richer passengers
Slight impact on survival

In [ ]:
#checking there is null value or not
df['Sex'].isnull().sum()

# replacing the null value


In [ ]:
#converting into number SEX
# so machine can easily understand it
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

In [ ]:
df.head()


In [ ]:
# checking we have missing value or not
df['Embarked'].isnull().sum()

In [ ]:
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True) # mode we will replace the value with most occuring value
                                                              # it is not numerical value or some kind of ....
## notes on mean and mode ...

In [ ]:
# again checking we have missing value or not
df['Embarked'].isnull().sum()

In [ ]:
df.head()


In [ ]:
# there are 2 things 1 is label encoding in which c s q will be 1 2 3 in this model will think 3 is better than 1 and 2 but in the case it is not like that
# we will use oneHot encoding in this it will think it as indiviual category
  # 0 0 c
  # 0 1 q
  # 1 0 s

df = pd.get_dummies(df, columns=['Embarked'], drop_first=True) # why we have use drop make a note on it

In [ ]:
df.head()

In [ ]:
df['Age'].isnull().sum()

In [ ]:
# we have used mean but little bit in diff way
  # we have not used on age as the only factor
  # we have used money as one catoregory(pclass) bec rich people will be older with family and all but poor will be like they are migrating young men

df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'].transform(
    lambda x: x.fillna(x.mean())
)

In [ ]:
df['Age'].isnull().sum()

🔥 When scaling is needed

✅ Needed for:

Logistic Regression
KNN
SVM

❌ Not needed for:

Decision Tree
Random Forest

In [ ]:
X = df.drop('Survived', axis=1)
y = df['Survived']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X_train[:5]
# this is for seeing how the data look after making all the value at one scale


In [ ]:
import pandas as pd

X_train_df = pd.DataFrame(X_train, columns=X.columns)
X_train_df.head()
# this is to see in the form of table

# train the model

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
# Accuracy check
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

In [ ]:
#Confusion Matrix
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

In [ ]:
#Detailed report
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

# Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

In [ ]:
y_pred_dt = dt_model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

dt_accuracy = accuracy_score(y_test, y_pred_dt)
print(dt_accuracy)

# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf_model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

rf_accuracy = accuracy_score(y_test, y_pred_rf)
print(rf_accuracy)

In [ ]:
# Improve random forest
rf_model = RandomForestClassifier(
    n_estimators=100,   # number of trees
    max_depth=5,        # control overfitting
    random_state=42
)

rf_model.fit(X_train, y_train)

In [ ]:
# Feature Importance
import pandas as pd

importance = rf_model.feature_importances_
feature_names = X.columns

feat_imp = pd.Series(importance, index=feature_names)
feat_imp.sort_values(ascending=False)

# 🧠 Ensemble Learning (core idea)

Definition (simple):
Combining multiple models to get better accuracy and stability than a single model.

In [ ]:
# now we will combines this
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

# Create models
lr = LogisticRegression()
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Combine models
voting_model = VotingClassifier(
    estimators=[
        ('lr', lr),
        ('dt', dt),
        ('rf', rf)
    ],
    voting='soft'   # better than hard voting
)

In [ ]:
from sklearn.metrics import accuracy_score

# Train
voting_model.fit(X_train, y_train)

# Predict
y_pred_vote = voting_model.predict(X_test)

# Accuracy
vote_acc = accuracy_score(y_test, y_pred_vote)
print("Voting Classifier Accuracy:", vote_acc)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

importance = rf_model.feature_importances_
feature_names = X.columns

feat_imp = pd.Series(importance, index=feature_names)
feat_imp.sort_values().plot(kind='barh')

plt.title("Feature Importance")
plt.show()